# Kaggle PCB Data Cleaning & SAM Point Extraction (GPU)

Runs on Kaggle GPU (T4 / P100) to clean bounding boxes and extract SAM polygon points across the full dataset.

### Settings:
- Accelerator: GPU T4 x2 or P100
- Internet: ON

In [ ]:
# 1. Install dependencies
!pip install -q opencv-python numpy pandas openpyxl matplotlib kagglehub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import json
import shutil
import cv2
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Active compute device:', device)
if device == 'cuda':
    print('GPU Model:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Download SAM Weights
import urllib.request
SAM_CHECKPOINT = Path('sam_vit_b.pth')
if not SAM_CHECKPOINT.exists():
    print('Downloading sam_vit_b.pth...')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth', str(SAM_CHECKPOINT))
    print('SAM weights ready.')

sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CHECKPOINT))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
print('SAM loaded on GPU!')

In [ ]:
# 3. Locate Dataset (Local or Kaggle)
import kagglehub

# Check if dataset is in local dir, Kaggle input, or download via kagglehub
possible_dirs = [
    Path('dataset_split/train/images'),
    Path('/kaggle/input/fics-pcb'),
    Path('/kaggle/input/pcb-component-detection')
]

images_dir = None
labels_dir = None

for p in possible_dirs:
    if p.exists():
        jpgs = list(p.rglob('*.jpg'))
        if jpgs:
            images_dir = jpgs[0].parent
            # labels are usually in sibling labels folder
            lbl_candidate = images_dir.parent / 'labels'
            if lbl_candidate.exists():
                labels_dir = lbl_candidate
            break

if images_dir is None:
    print('Fetching dataset from KaggleHub...')
    path = kagglehub.dataset_download('ficslab/fics-pcb')
    base_p = Path(path)
    jpgs = list(base_p.rglob('*.jpg'))
    if jpgs:
        images_dir = jpgs[0].parent
        labels_dir = images_dir.parent / 'labels'

print('Images Directory:', images_dir)
print('Labels Directory:', labels_dir)
all_images = sorted(list(images_dir.glob('*.jpg')))
print(f'Total images found: {len(all_images)}')

In [ ]:
# 4. Extraction & KiCad KLC Mapping Configuration
CLASS_MAP = {0: 'Cap1', 1: 'Cap2', 2: 'Cap3', 3: 'Cap4', 4: 'MOSFET', 5: 'Mov', 6: 'Resistor', 7: 'Transformer'}
PREFIX_MAP = {'Cap1': 'C', 'Cap2': 'C', 'Cap3': 'C', 'Cap4': 'C', 'MOSFET': 'Q', 'Mov': 'D', 'Resistor': 'R', 'Transformer': 'T'}
KICAD_FOOTPRINTS = {
    'Resistor': 'Resistor_SMD:R_0805_2012Metric',
    'Cap1': 'Capacitor_SMD:C_0805_2012Metric',
    'Cap2': 'Capacitor_SMD:C_1206_3216Metric',
    'Cap3': 'Capacitor_THT:CP_Radial_D6.3mm_P2.50mm',
    'Cap4': 'Capacitor_THT:CP_Radial_D8.0mm_P3.50mm',
    'MOSFET': 'Package_TO_SOT_SMD:SOT-23',
    'Mov': 'Diode_SMD:D_SOD-123',
    'Transformer': 'Transformer_SMD:Transformer_Bourns_SRF0703'
}

def clean_box(x1, y1, x2, y2, img_w, img_h, min_size=8):
    if x1 > x2: x1, x2 = x2, x1
    if y1 > y2: y1, y2 = y2, y1
    x1 = max(0, min(int(round(x1)), img_w - 1))
    y1 = max(0, min(int(round(y1)), img_h - 1))
    x2 = max(0, min(int(round(x2)), img_w))
    y2 = max(0, min(int(round(y2)), img_h))
    if (x2 - x1) < min_size or (y2 - y1) < min_size:
        return False, []
    return True, [x1, y1, x2, y2]

def extract_polygon_points(mask: np.ndarray, min_area: float = 15.0):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return []
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area: return []
    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True)
    return [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]

In [ ]:
# 5. Batch Process Dataset on GPU
output_dir = Path('kaggle_sam_output')
output_dir.mkdir(parents=True, exist_ok=True)
labels_out_dir = output_dir / 'labelme_json'
labels_out_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / 'kaggle_sam_points.csv'
xlsx_path = output_dir / 'kaggle_sam_points.xlsx'

records = []
total_boxes = 0

print(f'Starting GPU inference on {len(all_images)} images...')
for idx, img_p in enumerate(all_images, 1):
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    stem = img_p.stem
    lbl_p = (labels_dir / f'{stem}.txt') if labels_dir else None
    if not lbl_p or not lbl_p.exists(): continue
    
    boxes = []
    with open(lbl_p, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:5])
                ok, box = clean_box((xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h, w, h)
                if ok: boxes.append((box, cid))
    
    if not boxes: continue
    total_boxes += len(boxes)
    predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    
    shapes = []
    ref_counts = {}
    for inst_id, (box, cid) in enumerate(boxes, 1):
        masks, scores, _ = predictor.predict(box=np.array(box)[None, :], multimask_output=False)
        pts = extract_polygon_points(masks[0])
        if not pts:
            pts = [[float(box[0]), float(box[1])], [float(box[2]), float(box[1])], [float(box[2]), float(box[3])], [float(box[0]), float(box[3])]]
        
        cname = CLASS_MAP.get(cid, f'Class_{cid}')
        pfx = PREFIX_MAP.get(cname, 'U')
        ref_counts[pfx] = ref_counts.get(pfx, 0) + 1
        ref_des = f'{pfx}{ref_counts[pfx]}'
        
        records.append({
            'image': img_p.name, 'instance_id': inst_id, 'ref_des': ref_des, 'class': cname,
            'footprint': KICAD_FOOTPRINTS.get(cname, ''), 'confidence': round(float(scores[0]), 3),
            'points_count': len(pts), 'points_compact': '; '.join([f'({p[0]},{p[1]})' for p in pts]),
            'points_json': json.dumps(pts)
        })
        shapes.append({'label': f'{ref_des}: {cname}', 'points': pts, 'shape_type': 'polygon'})
    
    # Save LabelMe JSON
    with open(labels_out_dir / f'{stem}.json', 'w') as f:
        json.dump({'version': '5.5.0', 'flags': {}, 'shapes': shapes, 'imagePath': img_p.name, 'imageData': None, 'imageHeight': h, 'imageWidth': w}, f, indent=2)
    
    if idx % 50 == 0 or idx == len(all_images):
        pd.DataFrame(records).to_csv(csv_path, index=False)
        print(f'[{idx}/{len(all_images)}] Processed {len(records)} components...')

df = pd.DataFrame(records)
df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False)
print('Finished!')
print('Saved:', csv_path)
print('Saved:', xlsx_path)

In [ ]:
# 6. Package Outputs for 1-Click Download
shutil.make_archive('kaggle_sam_labelme_results', 'zip', output_dir)
print('Download zip ready: kaggle_sam_labelme_results.zip')